# Step 11: Validate the Model

**SageMaker Unified Studio Component**: Model Validation

Loads the registered model from MLflow, validates accuracy and prediction distribution against thresholds. Fails the workflow if the model does not meet quality gates.

In [ ]:
# Ensure sagemaker-mlflow plugin is installed for ARN-based tracking
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "sagemaker-mlflow", "--quiet", "--upgrade"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install failed: {result.stderr}")
else:
    print("sagemaker-mlflow installed/upgraded")

In [ ]:
# Parameters (injected by workflow via papermill)
mlflow_tracking_uri = "arn:aws:sagemaker:eu-west-1:146103651929:mlflow-tracking-server/machine-overheat-mlflow"
bucket_name = ""  # auto-detected below if not injected

In [ ]:
import pandas as pd
import mlflow
import boto3
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Auto-detect bucket if not injected by papermill
if not bucket_name:
    account_id = boto3.client('sts').get_caller_identity()['Account']
    bucket_name = f'sagemaker-unified-overheat-demo-{account_id}'
print(f"Using bucket: {bucket_name}")

## Setup MLflow

In [ ]:
try:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("machine-overheat")
    print(f"MLflow tracking URI: {mlflow_tracking_uri}")
except Exception as e:
    print(f"ARN-based tracking failed: {e}")
    import re
    match = re.search(r'arn:aws:sagemaker:([^:]+):[^:]+:mlflow-(?:app|tracking-server)/(.+)', mlflow_tracking_uri)
    if match:
        region, app_id = match.group(1), match.group(2)
        tracking_url = f"https://{app_id}.mlflow.sagemaker.{region}.app.aws"
        os.environ['MLFLOW_TRACKING_URI'] = tracking_url
        os.environ['MLFLOW_TRACKING_AWS_SIGV4'] = 'true'
        os.environ['AWS_DEFAULT_REGION'] = region
        mlflow.set_tracking_uri(tracking_url)
        mlflow.set_experiment("machine-overheat")
        print(f"MLflow set with URL fallback: {tracking_url}")
    else:
        raise

## Load Model from MLflow Registry

In [ ]:
model_name = "machine-overheat-model"
model = mlflow.sklearn.load_model(f"models:/{model_name}/latest")
print(f"\u2713 Loaded model: {model_name} (latest version)")

## Load Test Data

In [ ]:
# Recreate the same train/test split used during training (same random_state)
df = pd.read_parquet(f's3://{bucket_name}/data/features/features.parquet')
X = df[['temperature', 'temp_diff']]
y = df['overheat']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Test set: {len(X_test)} samples")

## Validation Checks

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Gate 1: Accuracy threshold
ACCURACY_THRESHOLD = 0.85
assert accuracy >= ACCURACY_THRESHOLD, (
    f"VALIDATION FAILED: Accuracy {accuracy:.4f} < threshold {ACCURACY_THRESHOLD}"
)
print(f"\u2713 Accuracy check passed: {accuracy:.4f} >= {ACCURACY_THRESHOLD}")

# Gate 2: Prediction distribution (not all one class)
pred_dist = pd.Series(y_pred).value_counts(normalize=True)
assert pred_dist.min() > 0.05, (
    f"VALIDATION FAILED: Model predicts only one class. Distribution: {pred_dist.to_dict()}"
)
print(f"\u2713 Distribution check passed: {pred_dist.to_dict()}")

# Gate 3: F1 score
F1_THRESHOLD = 0.80
assert f1 >= F1_THRESHOLD, (
    f"VALIDATION FAILED: F1 {f1:.4f} < threshold {F1_THRESHOLD}"
)
print(f"\u2713 F1 check passed: {f1:.4f} >= {F1_THRESHOLD}")

print("\n\u2713 All validation gates passed — model is ready for deployment")